# 03. Experimentación y Selección de Modelos

Con los datos limpios, enriquecidos y escalados, es hora de encontrar el algoritmo predictivo ganador.

### Instrucciones Generales:
1. **Validación:** No entrenes y midas sobre el mismo conjunto (sobreajuste). Recuerda haber dividido en Entrenamiento y Prueba antes.
2. **Entrenamiento Base:** Entrena los siguientes modelos base con tu set de Entrenamiento y compáralos usando RMSE (Error Cuadrático Medio):
   - `LinearRegression`
   - `SGDRegressor`
   - `DecisionTreeRegressor`
   - `RandomForestRegressor`
3. **Cross Validation (Validación Cruzada):** Para tener una métrica robusta, usa `cross_val_score` en el set de Entrenamiento para cada uno de los modelos anteriores.
4. **Ajuste Fino (Fine Tuning):** Toma el modelo ganador y busca sus mejores hiperparámetros. Utiliza un `GridSearchCV` explorando el número de estimadores (`n_estimators`), las características máximas (`max_features`), etc.
5. **Conclusión y Benchmark (IMPORTANTE):** Redacta una conclusión comparando los algoritmos. Explica por qué escogiste el modelo final y valida tu decisión calculando el RMSE sobre tu Set de Prueba que habías reservado. Documenta si alguno de tus modelos se sobreajusto o subajusto. Recuerda que el modelo final no puede tener esos problemas!


In [5]:
from pathlib import Path
import pandas as pd
import numpy as np
from sklearn.model_selection import KFold, cross_val_score
from sklearn.model_selection import GridSearchCV, KFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression, SGDRegressor
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor

In [10]:
# Librerías base


# Rutas de los datos procesados
train_path = Path("../data/processed/train_features_sin_log.csv")
test_path = Path("../data/processed/test_features_sin_log.csv")

# Cargar datasets
df_train = pd.read_csv(train_path)
df_test = pd.read_csv(test_path)

# Variable objetivo
target = "median_house_value"

# Separar predictoras y variable objetivo
X_train = df_train.drop(columns=[target])
y_train = df_train[target]

X_test = df_test.drop(columns=[target])
y_test = df_test[target]

# Alinear columnas de test con train
# Esto evita errores si alguna dummy no aparece en test o train
X_test = X_test.reindex(columns=X_train.columns, fill_value=0)

print("Train shape:", df_train.shape)
print("Test shape:", df_test.shape)
print("Número de variables predictoras:", X_train.shape[1])
print("Target:", target)
print("\nPrimeras columnas de X_train:")
print(X_train.columns.tolist()[:18])

Train shape: (16509, 18)
Test shape: (4128, 18)
Número de variables predictoras: 17
Target: median_house_value

Primeras columnas de X_train:
['longitude', 'latitude', 'housing_median_age', 'total_rooms', 'total_bedrooms', 'population', 'households', 'median_income', 'rooms_per_household', 'bedrooms_per_room', 'population_per_household', 'bedrooms_per_household', 'rooms_per_person', 'ocean_INLAND', 'ocean_ISLAND', 'ocean_NEAR BAY', 'ocean_NEAR OCEAN']


In [11]:
# Validación cruzada sobre el set de entrenamiento
cv = KFold(n_splits=5, shuffle=True, random_state=42)

# Modelos base
modelos = {
    "LinearRegression": Pipeline([
        ("scaler", StandardScaler()),
        ("model", LinearRegression())
    ]),

    "SGDRegressor": Pipeline([
        ("scaler", StandardScaler()),
        ("model", SGDRegressor(random_state=42, max_iter=2000, tol=1e-3))
    ]),

    "DecisionTreeRegressor": DecisionTreeRegressor(random_state=42),

    "RandomForestRegressor": RandomForestRegressor(
        random_state=42,
        n_estimators=200,
        n_jobs=-1
    ),

    "GradientBoostingRegressor": GradientBoostingRegressor(
        random_state=42,
        n_estimators=200,
        learning_rate=0.05,
        max_depth=3
    ),

    "XGBRegressor": XGBRegressor(
        random_state=42,
        n_estimators=300,
        learning_rate=0.05,
        max_depth=4,
        subsample=0.8,
        colsample_bytree=0.8,
        objective="reg:squarederror",
        n_jobs=-1
    ),

    "LGBMRegressor": LGBMRegressor(
        random_state=42,
        n_estimators=300,
        learning_rate=0.05,
        num_leaves=31,
        subsample=0.8,
        colsample_bytree=0.8,
        n_jobs=-1
    )
}

In [12]:
# Evaluar modelos base con RMSE en validación cruzada
resultados = []

for nombre, modelo in modelos.items():
    scores = cross_val_score(
        modelo,
        X_train,
        y_train,
        cv=cv,
        scoring="neg_root_mean_squared_error",
        n_jobs=-1
    )
    
    resultados.append({
        "modelo": nombre,
        "rmse_cv_mean": -scores.mean(),
        "rmse_cv_std": scores.std()
    })

resultados_df = pd.DataFrame(resultados).sort_values("rmse_cv_mean").reset_index(drop=True)
display(resultados_df)

,modelo,rmse_cv_mean,rmse_cv_std
0,LGBMRegressor,4.501765e+04,1.158526e+03
1,XGBRegressor,4.790596e+04,1.228031e+03
2,RandomForestRegressor,4.971362e+04,1.420345e+03
3,GradientBoostingRegressor,5.215106e+04,1.294192e+03
4,LinearRegression,6.458663e+04,1.538987e+03
5,DecisionTreeRegressor,7.121333e+04,1.199192e+03
6,SGDRegressor,2.130847e+09,1.698446e+09


In [20]:
# Mostrar el modelo líder provisional
mejor_modelo = resultados_df.loc[0, "modelo"]
mejor_rmse = resultados_df.loc[0, "rmse_cv_mean"]

print(f"Modelo líder provisional: {mejor_modelo}")
print(f"RMSE promedio en validación cruzada: {mejor_rmse:,.2f}")

Modelo líder provisional: LGBMRegressor
RMSE promedio en validación cruzada: 45,017.65


### Ajuste fino (Fine Tuning) Random Forest


In [21]:
#Definir validación cruzada

cv = KFold(n_splits=5, shuffle=True, random_state=42)

In [22]:
# modelo base
rf = RandomForestRegressor(
    random_state=42,
    n_jobs=-1
)

In [ ]:
param_grid = {
    "n_estimators": [100, 200, 300],
    "max_depth": [None, 10, 20, 30],
    "min_samples_split": [2, 5, 10],
    "min_samples_leaf": [1, 2, 4],
    "max_features": ["sqrt", "log2"]
}

In [ ]:
grid_search = GridSearchCV(
    estimator=rf,
    param_grid=param_grid,
    scoring="neg_root_mean_squared_error",
    cv=cv,
    n_jobs=-1,
    verbose=1
)

grid_search.fit(X_train, y_train)

In [ ]:
resultados_grid = pd.DataFrame(grid_search.cv_results_)

resultados_grid["rmse_cv_mean"] = -resultados_grid["mean_test_score"]
resultados_grid["rmse_cv_std"] = resultados_grid["std_test_score"]

resultados_grid = resultados_grid.sort_values("rmse_cv_mean").reset_index(drop=True)

display(
    resultados_grid[
        ["params", "rmse_cv_mean", "rmse_cv_std", "rank_test_score"]
    ].head(10)
)

,params,rmse_cv_mean,rmse_cv_std,rank_test_score
0,"{'max_depth': 30, 'max_features': 'sqrt', 'min...",52180.054498,1300.196306,1
1,"{'max_depth': 30, 'max_features': 'log2', 'min...",52180.054498,1300.196306,1
2,"{'max_depth': None, 'max_features': 'sqrt', 'm...",52241.269814,1299.625508,3
3,"{'max_depth': None, 'max_features': 'log2', 'm...",52241.269814,1299.625508,3
4,"{'max_depth': 20, 'max_features': 'sqrt', 'min...",52241.446523,1307.448971,5
5,"{'max_depth': 20, 'max_features': 'log2', 'min...",52241.446523,1307.448971,5
6,"{'max_depth': 30, 'max_features': 'sqrt', 'min...",52245.403773,1302.316816,7
7,"{'max_depth': 30, 'max_features': 'log2', 'min...",52245.403773,1302.316816,7
8,"{'max_depth': None, 'max_features': 'log2', 'm...",52258.186871,1318.431749,9
9,"{'max_depth': None, 'max_features': 'sqrt', 'm...",52258.186871,1318.431749,9


In [ ]:
best_rf = grid_search.best_estimator_
best_rf

,n_estimators,300
,criterion,'squared_error'
,max_depth,30
,min_samples_split,2
,min_samples_leaf,1
,min_weight_fraction_leaf,0.0
,max_features,'sqrt'
,max_leaf_nodes,None
,min_impurity_decrease,0.0
,bootstrap,True
,oob_score,False


In [25]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, r2_score
import numpy as np

# Modelo base
rf_base = RandomForestRegressor(
    random_state=42,
    n_estimators=200,
    n_jobs=-1
)

# Modelo ajustado (el mejor del GridSearch)
rf_tuned = RandomForestRegressor(
    n_estimators=300,
    max_depth=30,
    max_features='sqrt',
    min_samples_split=2,
    min_samples_leaf=1,
    random_state=42,
    n_jobs=-1
)

# Entrenar ambos con todo el train
rf_base.fit(X_train, y_train)
rf_tuned.fit(X_train, y_train)

# Predicciones en test
y_pred_base = rf_base.predict(X_test)
y_pred_tuned = rf_tuned.predict(X_test)

# Métricas
rmse_base = np.sqrt(mean_squared_error(y_test, y_pred_base))
rmse_tuned = np.sqrt(mean_squared_error(y_test, y_pred_tuned))

r2_base = r2_score(y_test, y_pred_base)
r2_tuned = r2_score(y_test, y_pred_tuned)

print("=== Resultados en TEST ===")
print(f"RandomForest base  -> RMSE: {rmse_base:,.2f} | R2: {r2_base:.4f}")
print(f"RandomForest tuned -> RMSE: {rmse_tuned:,.2f} | R2: {r2_tuned:.4f}")

=== Resultados en TEST ===
RandomForest base  -> RMSE: 49,999.61 | R2: 0.8132
RandomForest tuned -> RMSE: 50,091.00 | R2: 0.8126


### Ajuste fino (Fine Tuning) sobre LGBMRegressor


In [ ]:
from lightgbm import LGBMRegressor
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import mean_squared_error, r2_score
import numpy as np
import pandas as pd

# Modelo base
lgbm = LGBMRegressor(
    objective="regression",
    random_state=42,
    n_jobs=-1,
    verbosity=-1
)

# Malla reducida y útil para precio de viviendas
param_grid_lgbm = {
    "n_estimators": [200, 400],
    "learning_rate": [0.03, 0.05, 0.1],
    "num_leaves": [31, 50],
    "max_depth": [-1, 10, 20],
    "min_child_samples": [10, 20, 30],
    "subsample": [0.8, 1.0],
    "colsample_bytree": [0.8, 1.0]
}

grid_lgbm = GridSearchCV(
    estimator=lgbm,
    param_grid=param_grid_lgbm,
    scoring="neg_root_mean_squared_error",
    cv=5,
    n_jobs=-1,
    verbose=1
)

grid_lgbm.fit(X_train, y_train)

print("Mejores parámetros encontrados:")
print(grid_lgbm.best_params_)

print(f"\nMejor RMSE CV: {-grid_lgbm.best_score_:,.2f}")

In [ ]:
best_lgbm = grid_lgbm.best_estimator_

y_pred_lgbm = best_lgbm.predict(X_test)

mse_lgbm = mean_squared_error(y_test, y_pred_lgbm)
rmse_lgbm = np.sqrt(mse_lgbm)
r2_lgbm = r2_score(y_test, y_pred_lgbm)

print("=== Resultados en TEST ===")
print(f"LGBM tuned -> RMSE: {rmse_lgbm:,.2f} | R2: {r2_lgbm:.4f}")

=== Resultados en TEST ===
LGBM tuned -> RMSE: 44,623.85 | R2: 0.8512


-----------

In [ ]:
from lightgbm import LGBMRegressor
from sklearn.model_selection import GridSearchCV, KFold

cv = KFold(n_splits=5, shuffle=True, random_state=42)

lgbm = LGBMRegressor(
    random_state=42,
    n_jobs=-1
)

param_grid_lgbm_small = {
    "n_estimators": [300, 400, 500],
    "learning_rate": [0.03, 0.05],
    "num_leaves": [31, 40, 50],
    "max_depth": [-1, 10],
    "min_child_samples": [15, 20, 25],
    "subsample": [0.8, 0.9],
    "colsample_bytree": [0.7, 0.8],
    "reg_alpha": [0.0, 0.1],
    "reg_lambda": [0.0, 0.1, 0.5]
}

grid_lgbm_small = GridSearchCV(
    estimator=lgbm,
    param_grid=param_grid_lgbm_small,
    scoring="neg_root_mean_squared_error",
    cv=cv,
    n_jobs=-1,
    verbose=1
)

grid_lgbm_small.fit(X_train, y_train)

print("Mejores parámetros:", grid_lgbm_small.best_params_)
print(f"Mejor RMSE CV: {-grid_lgbm_small.best_score_:,.2f}")

Fitting 5 folds for each of 2592 candidates, totalling 12960 fits
[LightGBM] [Warning] Found whitespace in feature_names, replace with underlines
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.002202 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 3119
[LightGBM] [Info] Number of data points in the train set: 16509, number of used features: 16
[LightGBM] [Info] Start training from score 206333.427827
Mejores parámetros: {'colsample_bytree': 0.8, 'learning_rate': 0.05, 'max_depth': -1, 'min_child_samples': 15, 'n_estimators': 500, 'num_leaves': 50, 'reg_alpha': 0.1, 'reg_lambda': 0.0, 'subsample': 0.8}
Mejor RMSE CV: 44,053.36


In [ ]:
from sklearn.metrics import mean_squared_error, r2_score
import numpy as np

# Mejor modelo encontrado
best_lgbm_small = grid_lgbm_small.best_estimator_

# Predicción sobre test
y_pred_lgbm_small = best_lgbm_small.predict(X_test)

# Métricas
mse_lgbm_small = mean_squared_error(y_test, y_pred_lgbm_small)
rmse_lgbm_small = np.sqrt(mse_lgbm_small)
r2_lgbm_small = r2_score(y_test, y_pred_lgbm_small)

print("=== Resultados en TEST ===")
print(f"LGBM tuned final -> RMSE: {rmse_lgbm_small:,.2f} | R2: {r2_lgbm_small:.4f}")

=== Resultados en TEST ===
LGBM tuned final -> RMSE: 44,513.67 | R2: 0.8520


### Benchmark y Conclusión Final
*(Escribe tus conclusiones de negocio aquí)*

In [26]:

# Diccionario de modelos finales
modelos_finales = {
    "LinearRegression": modelos["LinearRegression"].fit(X_train, y_train),
    "SGDRegressor": modelos["SGDRegressor"].fit(X_train, y_train),
    "DecisionTree": DecisionTreeRegressor(random_state=42).fit(X_train, y_train),
    "RandomForest_tuned": rf_tuned,
    "LGBM_final": best_lgbm_small
}

# Lista para resultados
resultados_test = []

# Evaluar cada modelo
for nombre, modelo in modelos_finales.items():
    y_pred = modelo.predict(X_test)
    
    mse = mean_squared_error(y_test, y_pred)
    rmse = np.sqrt(mse)
    r2 = r2_score(y_test, y_pred)
    
    resultados_test.append({
        "modelo": nombre,
        "RMSE": rmse,
        "R2": r2
    })

# Convertir a DataFrame
resultados_test_df = pd.DataFrame(resultados_test).sort_values("RMSE").reset_index(drop=True)

display(resultados_test_df)

,modelo,RMSE,R2
0,LGBM_final,4.451367e+04,8.519742e-01
1,RandomForest_tuned,5.009100e+04,8.125568e-01
2,LinearRegression,6.672202e+04,6.674256e-01
3,DecisionTree,6.996254e+04,6.343366e-01
4,SGDRegressor,2.578210e+10,-4.965776e+10


- Los diferentes escenarios de preprocesamiento evaluados generaron resultados muy similares en desempeño, lo que indica que la información más valiosa para predecir el precio de la vivienda ya estaba contenida en las variables originales del dataset.

- Variables como el ingreso medio del sector, la ubicación geográfica y la antigüedad de las viviendas concentran gran parte del poder explicativo del problema, por lo que agregar más transformaciones no produjo mejoras sustanciales.

- La incorporación de variables derivadas y transformaciones logarítmicas aportó orden y robustez al proceso analítico, pero no cambió de manera significativa la capacidad predictiva del modelo final.

- Desde una perspectiva de negocio, esto significa que el valor de una vivienda en este conjunto de datos está determinado principalmente por factores estructurales del entorno y no tanto por ajustes menores en la preparación de los datos.

- El mejor modelo alcanzó un RMSE de 44,513.67 dólares y un R² de 0.8520, lo que indica que el sistema logra explicar aproximadamente el 85.2 % de la variabilidad del precio de la vivienda.

- En términos prácticos, el modelo presenta un error promedio de alrededor de 44.5 mil dólares por predicción. Aunque este valor puede parecer alto en términos absolutos, debe interpretarse considerando que el precio de las viviendas del dataset varía desde aproximadamente 14,999 dólares hasta 500,001 dólares.

- En viviendas de menor valor, este margen de error representa una proporción alta del precio real, mientras que en viviendas de valor medio o alto su impacto relativo es considerablemente menor. Por ello, el modelo resulta más útil como herramienta de estimación general y apoyo a la toma de decisiones que como mecanismo de tasación exacta a nivel individual.

- El hecho de que distintas versiones del dataset hayan producido métricas casi iguales sugiere que el proyecto alcanzó un nivel de desempeño cercano al máximo posible con la información disponible.
- No fue posible reducir mucho más el error porque la base de datos no contiene variables más detalladas del inmueble, como estado de conservación, acabados, remodelaciones, número real de baños, garaje, vista o características específicas del vecindario, factores que en la práctica influyen de forma importante en el precio.

- En consecuencia, la principal oportunidad de mejora futura no está tanto en seguir ajustando el mismo modelo, sino en enriquecer la base de datos con variables más representativas del valor real de cada vivienda.

- Desde el punto de vista del negocio, el modelo final sí es útil para identificar tendencias de precio, comparar zonas, estimar rangos de valor y apoyar análisis inmobiliarios a escala general, pero no reemplaza una valoración individual detallada cuando se requiere alta precisión.

- Al comparar predicciones individuales, se observa que el modelo no se comporta igual en todas las viviendas, sino que su precisión depende de las características de cada inmueble y del segmento de precio al que pertenece.

-En una vivienda con valor real de 96,100 dólares, el modelo estimó aproximadamente 110,160 dólares, lo que representa una sobreestimación visible. Esto muestra que, en inmuebles de menor valor, el error relativo puede ser más sensible y tener mayor impacto en la interpretación del resultado.

-En cambio, para una vivienda con valor real de 458,300 dólares, el modelo predijo aproximadamente 448,817 dólares, lo que refleja una diferencia mucho menor en términos porcentuales. Este comportamiento sugiere que, en ciertos casos, el modelo logra capturar de mejor manera la lógica del mercado y aproximarse con mayor precisión al precio observado.

- Estos resultados confirman que el modelo sí aprende patrones relevantes de la información disponible, ya que no genera estimaciones aleatorias y logra acercarse razonablemente al valor real en múltiples casos.

- Sin embargo, también evidencian que el desempeño no es uniforme para todas las observaciones. Por ello, el modelo debe entenderse como una herramienta de apoyo para estimar precios de vivienda, mas no como un mecanismo de tasación exacta.

- En términos de negocio, el modelo aporta valor porque permite generar estimaciones automáticas con un nivel de ajuste aceptable, lo que puede ser útil para análisis preliminar, priorización de inmuebles y apoyo en decisiones comerciales. Aun así, su margen de error indica que la predicción final siempre debe complementarse con criterio experto y con variables adicionales que no están presentes en el dataset.